# Signals of Prestige in Film Discourse: A Transformer-Based Approach to Predicting Best Picture Winners

### By: Ed Hou, Si Qin Huang, Alejandro Mendez

---

Click here to [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/s-huang23/nlp-oscar-predictor/blob/main/final_oscar_predictor.ipynb)


<font color = 'orange'>**Preface:**

This is the cleaned up final notebook with the model outputs. For more details on model development, checkout model_development.ipynb (all three models) on GitHub. </font>

In [ ]:
!pip install kagglehub sentence-transformers

In [ ]:
import kagglehub
import os
import glob
import json
import pandas as pd
import numpy as np
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel, AutoModel, AutoTokenizer
from typing import Optional

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import scipy.sparse as sp

In [ ]:
# Mount drive (If manually uploaded data, mount drive)
# from google.colab import drive
# drive.mount('/content/drive')

# Clone only the data folder from the repository (if not already present)
if not os.path.exists("data"):
    os.system("rm -rf /tmp/nlp-repo")
    ret = os.system("git clone --depth=1 https://github.com/s-huang23/nlp-oscar-predictor.git /tmp/nlp-repo")
    if ret == 0:
        os.system("cp -r /tmp/nlp-repo/data .")
        print("Data cloned successfully.")
    else:
        print("ERROR: git clone failed. Check that the repo is public and the URL is correct.")

print("data/ exists:", os.path.exists("data"))
print("data/raw/ exists:", os.path.exists("data/raw"))

Data cloned successfully.
data/ exists: True
data/raw/ exists: True


## Dataset

Load web-scraped reviews from Metacritic and iMDb data from kaggle

#### Metacritic data

In [ ]:
# Load metacritic dataset
# metacritic_df = pd.read_csv("/content/drive/MyDrive/metacritic_reviews.csv") # If data is uploaded to drive
metacritic_df = pd.read_csv("data/raw/metacritic_reviews.csv")
print(metacritic_df.shape)
metacritic_df.head()

(6643, 13)


,ceremony_year,release_year,film_title,winner,metacritic_slug,metacritic_url,critic_review_page,review_date,critic_score,publication,author,quote,full_review_url
0,2010,2009,Avatar,0,avatar,https://www.metacritic.com/movie/avatar/,https://www.metacritic.com/movie/avatar/critic...,NaN,100,The Hollywood Reporter,Kirk Honeycutt,"A fully believable, flesh-and-blood (albeit no...",http://www.hollywoodreporter.com/hr/film-revie...
1,2010,2009,Avatar,0,avatar,https://www.metacritic.com/movie/avatar/,https://www.metacritic.com/movie/avatar/critic...,NaN,100,Empire,Chris Hewitt (1),"It's been twelve years since ""Titanic,"" but th...",http://www.empireonline.com/reviews/reviewcomp...
2,2010,2009,Avatar,0,avatar,https://www.metacritic.com/movie/avatar/,https://www.metacritic.com/movie/avatar/critic...,NaN,90,Variety,Todd McCarthy,"Avatar is all-enveloping and transporting, wit...",http://www.variety.com/review/VE1117941773.htm...
3,2010,2009,Avatar,0,avatar,https://www.metacritic.com/movie/avatar/,https://www.metacritic.com/movie/avatar/critic...,NaN,100,Chicago Sun-Times,Roger Ebert,"Once again, [Cameron] has silenced the doubter...",http://rogerebert.suntimes.com/apps/pbcs.dll/a...
4,2010,2009,Avatar,0,avatar,https://www.metacritic.com/movie/avatar/,https://www.metacritic.com/movie/avatar/critic...,NaN,75,Chicago Tribune,Michael Phillips,The first 90 minutes of Avatar are pretty terr...,http://featuresblogs.chicagotribune.com/talkin...


#### Kaggle iMDb data

In [ ]:
# Load nominees dataset
# nominees_df = pd.read_csv("/content/drive/MyDrive/nominees.csv") # If data is uploaded to drive
nominees_df   = pd.read_csv("data/nominees.csv")
print(nominees_df.shape)
nominees_df.head()

(136, 5)


,ceremony_year,film_title,release_year,winner,metacritic_slug
0,2010,Avatar,2009,0,avatar
1,2010,The Blind Side,2009,0,the-blind-side
2,2010,District 9,2009,0,district-9
3,2010,An Education,2009,0,an-education
4,2010,The Hurt Locker,2009,1,the-hurt-locker


In [ ]:
# Download latest version of data from kaggle
path = kagglehub.dataset_download("ebiswas/imdb-review-dataset")

print("Path to dataset files:", path)
# print(os.listdir(path))

100%|██████████| 2.69G/2.69G [00:20<00:00, 138MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ebiswas/imdb-review-dataset/versions/1


In [ ]:
# Keep Metacritic / nominee titles as the canonical film titles.
# Kaggle IMDb titles sometimes include alternate punctuation, accents, or IMDb roman-numeral suffixes.
# These variants are mapped back to the Metacritic title before merging.
imdb_to_metacritic_title = {
    "Extremely Loud & Incredibly Close": "Extremely Loud and Incredibly Close",
    "Les Misérables": "Les Miserables",
    "Once Upon a Time... In Hollywood": "Once Upon a Time ... in Hollywood",
    "Hell or High Water (II)": "Hell or High Water",
    "Boyhood (I)": "Boyhood",
    "Arrival (II)": "Arrival",
    "Get Out (I)": "Get Out",
    "Moonlight (I)": "Moonlight",
    "Room (I)": "Room",
    "Spotlight (I)": "Spotlight",
    "The Artist (I)": "The Artist",
    "Vice (I)": "Vice"
}

# nominees_df and metacritic_df already use canonical Metacritic titles.
# Apply imdb_to_metacritic_title to reviews_df["movie"] after extracting IMDb movie titles.


In [ ]:
# Combine all parts of json
all_files = glob.glob(os.path.join(path, "part-*.json"))

records = []
for f in all_files:
    with open(f) as fh:
        records.extend(json.load(fh))

# Convert to dataframe
reviews_df = pd.DataFrame(records)
print(reviews_df.shape)
reviews_df.head()

(5571499, 9)


,review_id,reviewer,movie,rating,review_summary,review_date,spoiler_tag,review_detail,helpful
0,rw1133942,OriginalMovieBuff21,Kill Bill: Vol. 2 (2004),8,Good follow up that answers all the questions,24 July 2005,0,"After seeing Tarantino's Kill Bill Vol: 1, I g...","[0, 1]"
1,rw1133943,sentra14,Journey to the Unknown (1968â€“ ),None,Excellent series,24 July 2005,0,"I have the entire series on video, taped mostl...","[11, 11]"
2,rw1133946,GreenwheelFan2002,The Island (2005),9,"Not just about action, but about survival...",24 July 2005,0,Once again the critics prove themselves as mor...,"[2, 5]"
3,rw1133948,itsascreambaby,Win a Date with Tad Hamilton! (2004),3,Falls under the category: seen it a million ti...,24 July 2005,0,This IS a film that has been done too many tim...,"[2, 3]"
4,rw1133949,OriginalMovieBuff21,Saturday Night Live: The Best of Chris Farley ...,10,"Before Tommy Boy and Black Sheep, there was Sa...",24 July 2005,0,Chris Farley is one of my favorite comedians a...,"[4, 4]"


In [ ]:
# Extract title and year from "Movie Title (YEAR)" format
reviews_df["release_year"] = (
    reviews_df["movie"]
    .str.extract(r'\((\d{4})')   # grab the first 4-digit year after '('
    .astype(float)
    .astype("Int64")
)

reviews_df["movie"] = (
    reviews_df["movie"]
    .str.replace(r'\s*\(\d{4}[^)]*\)', '', regex=True)  # remove anything from (YEAR...  to )
    .str.strip()
)

# Convert Kaggle IMDb title variants to canonical Metacritic / nominee titles.
reviews_df["movie"] = reviews_df["movie"].replace(imdb_to_metacritic_title)

print(reviews_df[["movie", "release_year"]].head(10))

                                           movie  release_year
0                              Kill Bill: Vol. 2          2004
1                         Journey to the Unknown          1968
2                                     The Island          2005
3                  Win a Date with Tad Hamilton!          2004
4  Saturday Night Live: The Best of Chris Farley          2000
5                                    Outlaw Star          1998
6                                    The Aviator          2004
7      Star Wars: Episode I - The Phantom Menace          1999
8                          The Amityville Horror          2005
9                                  Flying Tigers          1942


In [ ]:
# Filter and merge using both columns
filtered_reviews = reviews_df[
    reviews_df["movie"].isin(nominees_df["film_title"]) &
    reviews_df["release_year"].isin(nominees_df["release_year"])
].copy()

print(f"Total reviews matched: {len(filtered_reviews)}")
print(f"Unique films matched : {filtered_reviews['movie'].nunique()} / {len(nominees_df)}")

imdb_df = filtered_reviews.merge(
    nominees_df[["film_title", "release_year", "ceremony_year", "winner"]],
    left_on=["movie", "release_year"],
    right_on=["film_title", "release_year"],
    how="left"
)

# Drop values with NAs that don't match up
imdb_df = imdb_df.dropna(subset=["winner"])
# Fix winner and ceremony_year column formatting
imdb_df["winner"] = imdb_df["winner"].astype(int)
imdb_df["ceremony_year"] = imdb_df["ceremony_year"].astype(int)

print(imdb_df.shape)
imdb_df.head()

Total reviews matched: 84642
Unique films matched : 87 / 96
(84107, 13)


,review_id,reviewer,movie,rating,review_summary,review_date,spoiler_tag,review_detail,helpful,release_year,film_title,ceremony_year,winner
0,rw2531251,galvanekps,The Help,8,You just never know...,12 December 2011,0,I have to state up front I went into watching ...,"[0, 5]",2011,The Help,2012,0
1,rw2531254,remedy305,Hugo,8,1920's Paris comes to life,12 December 2011,0,"This is a wonderful story about a boy, Hugo, w...","[4, 6]",2011,Hugo,2012,0
2,rw2531322,gnguyen102,Midnight in Paris,9,Well-written script and outstanding acting,12 December 2011,0,I must admit that I have been a fan of Woody A...,"[0, 1]",2011,Midnight in Paris,2012,0
3,rw2531410,mmobini,Hugo,7,A huge potential for an exceptional film.,12 December 2011,0,A resplendently sweeping opening shot that pro...,"[2, 5]",2011,Hugo,2012,0
4,rw2531412,Abigail_fox,Hugo,1,Couldn't wait for it to end - so dull!,12 December 2011,0,I saw this film yesterday. I didn't see it in ...,"[88, 176]",2011,Hugo,2012,0


In [ ]:
# Drop unneeded columns
imdb_df = imdb_df.drop(columns=["movie", "review_summary", "spoiler_tag", "helpful"])

# Strip disambiguation suffixes like (I), (II) from film titles.
for df in [imdb_df, nominees_df, metacritic_df]:
    df["film_title"] = df["film_title"].str.replace(r'\s*\([IV]+\)$', '', regex=True).str.strip()

# Rating is useful for possible later features; keep it numeric.
imdb_df["rating"] = pd.to_numeric(imdb_df["rating"], errors="coerce")

# Keep the model scope used downstream in this notebook.
nominees_df_filtered = nominees_df[
    (nominees_df["ceremony_year"] >= 2012) &
    (nominees_df["ceremony_year"] <= 2020)
].copy()

print(imdb_df.columns.tolist())
print(imdb_df["winner"].value_counts())
imdb_df.head()

['review_id', 'reviewer', 'rating', 'review_date', 'review_detail', 'release_year', 'film_title', 'ceremony_year', 'winner']
winner
0    73552
1    10555
Name: count, dtype: int64


,review_id,reviewer,rating,review_date,review_detail,release_year,film_title,ceremony_year,winner
0,rw2531251,galvanekps,8,12 December 2011,I have to state up front I went into watching ...,2011,The Help,2012,0
1,rw2531254,remedy305,8,12 December 2011,"This is a wonderful story about a boy, Hugo, w...",2011,Hugo,2012,0
2,rw2531322,gnguyen102,9,12 December 2011,I must admit that I have been a fan of Woody A...,2011,Midnight in Paris,2012,0
3,rw2531410,mmobini,7,12 December 2011,A resplendently sweeping opening shot that pro...,2011,Hugo,2012,0
4,rw2531412,Abigail_fox,1,12 December 2011,I saw this film yesterday. I didn't see it in ...,2011,Hugo,2012,0


## Pre-processing

Both IMDb and Metacritic reviews are processed with the same core leakage-control and cleanup steps.

Implemented steps:
  - Filter to Oscar ceremony years 2012-2020
  - Filter each review to before the ceremony date: `review_date < ceremony_date`
  - Remove rows with invalid dates, short or empty text, and duplicate review text
  - Clean review text into `clean_text` while preserving emoji and sentiment-bearing punctuation
  - Add `film_review_count` and `film_review_weight` so films with more reviews do not dominate downstream aggregation

The ceremony date is excluded because review timestamps are date-only; excluding Oscar day avoids accidentally including reviews written after the winner was announced.


In [ ]:
# Preprocess IMDb and Metacritic reviews.
# Core window: review_date < ceremony_date.
import re

START_YEAR = 2012
END_YEAR = 2020
MIN_TEXT_CHARS = 20

# windows_path = "/content/drive/MyDrive/oscar_windows.csv"
oscar_windows = pd.read_csv("data/oscar_windows.csv", parse_dates=["nomination_date", "ceremony_date"])


def clean_text(value):
    """Normalize review text while preserving the words used by reviewers."""
    text = "" if pd.isna(value) else str(value)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# Sanity check: emoji can carry sentiment, so preprocessing should keep them.
assert clean_text("Loved it 😭!") == "Loved it 😭!"


def summarize_reviews(label, df):
    film_count = df[["ceremony_year", "film_title"]].drop_duplicates().shape[0]
    years = sorted(df["ceremony_year"].dropna().astype(int).unique().tolist()) if len(df) else []
    print(f"{label:<36} rows={len(df):>6} films={film_count:>3} years={years}")


def preprocess_reviews(df, source_name, text_col):
    """Apply project preprocessing to one review source."""
    print(f"\n{source_name.upper()} preprocessing")
    summarize_reviews("raw", df)

    working = df[
        (df["ceremony_year"] >= START_YEAR) &
        (df["ceremony_year"] <= END_YEAR)
    ].copy()
    summarize_reviews("after ceremony-year filter", working)

    working = working.merge(oscar_windows, on="ceremony_year", how="left", validate="many_to_one")
    working["review_date_parsed"] = pd.to_datetime(working["review_date"], errors="coerce")

    invalid_dates = working["review_date_parsed"].isna().sum()
    if invalid_dates:
        print(f"dropped invalid review dates: {invalid_dates}")

    in_window = (
        working["review_date_parsed"].notna()
        & (working["review_date_parsed"] < working["ceremony_date"])
    )
    working = working[in_window].copy()
    summarize_reviews("after ceremony-date filter", working)

    working["clean_text"] = working[text_col].map(clean_text)
    before = len(working)
    working = working[working["clean_text"].str.len() >= MIN_TEXT_CHARS].copy()
    print(f"dropped short/empty reviews: {before - len(working)}")

    dedupe_cols = ["ceremony_year", "film_title", "clean_text"]
    if source_name == "imdb" and "reviewer" in working.columns:
        dedupe_cols.insert(2, "reviewer")
    if source_name == "metacritic" and "publication" in working.columns:
        dedupe_cols.insert(2, "publication")

    before = len(working)
    working = working.drop_duplicates(subset=dedupe_cols).copy()
    print(f"dropped duplicate reviews: {before - len(working)}")

    working["film_review_count"] = working.groupby(
        ["ceremony_year", "film_title"]
    )["clean_text"].transform("size")
    working["film_review_weight"] = 1.0 / working["film_review_count"]

    summarize_reviews("final", working)
    return working


imdb_df = preprocess_reviews(imdb_df, "imdb", "review_detail")
metacritic_df = preprocess_reviews(metacritic_df, "metacritic", "quote")

## Model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from sentence_transformers import SentenceTransformer
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

# ── PRECOMPUTE WITH SENTENCE TRANSFORMERS ─────────────────────────────────────
def precompute_st_embeddings(nominees_df, metacritic_df, imdb_df,
                              max_critic_reviews=40, max_imdb_reviews=50,
                              save_path="/content/drive/MyDrive/oscar_st_embeddings.pt",
                              device="cuda"):
    """
    Encode all reviews using sentence-transformers all-mpnet-base-v2.
    Produces genuinely discriminative embeddings — different films
    will have meaningfully different vectors.
    Saves to Drive so you never recompute.
    """
    print("Loading sentence transformer...")
    st_model = SentenceTransformer("all-mpnet-base-v2", device=device)
    st_model.eval()

    cache = {}

    for year, year_nominees in nominees_df.groupby("ceremony_year"):
        print(f"\nEncoding year {year}...")
        films      = year_nominees["film_title"].tolist()
        winner_row = year_nominees[year_nominees["winner"] == 1]

        if winner_row.empty:
            print(f"  Warning: no winner for {year}, skipping")
            continue

        winner_title = winner_row["film_title"].iloc[0]
        if winner_title not in films:
            print(f"  Warning: winner not in films for {year}, skipping")
            continue

        winner_idx = films.index(winner_title)
        cache[year] = []

        for film in films:
            # Get critic reviews
            critic_texts = metacritic_df[
                (metacritic_df["film_title"]    == film) &
                (metacritic_df["ceremony_year"] == year)
            ]["clean_text"].dropna().tolist()[:max_critic_reviews]

            # Get IMDb reviews
            imdb_texts = imdb_df[
                (imdb_df["film_title"]    == film) &
                (imdb_df["ceremony_year"] == year)
            ]["clean_text"].dropna().tolist()[:max_imdb_reviews]

            # Fallbacks
            if not critic_texts:
                print(f"  Warning: no critic reviews for {film}")
                critic_texts = ["no critic reviews available"]
            if not imdb_texts:
                print(f"  Warning: no IMDb reviews for {film}")
                imdb_texts = ["no user reviews available"]

            # Encode — sentence transformers handle batching internally
            with torch.no_grad():
                c_emb = st_model.encode(
                    critic_texts,
                    convert_to_tensor=True,
                    show_progress_bar=False
                )  # [num_critic_reviews, 768]

                i_emb = st_model.encode(
                    imdb_texts,
                    convert_to_tensor=True,
                    show_progress_bar=False
                )  # [num_imdb_reviews, 768]

            # Pad to max_reviews so tensors are uniform shape
            c_emb = pad_embeddings(c_emb, max_critic_reviews)  # [max_critic, 768]
            i_emb = pad_embeddings(i_emb, max_imdb_reviews)    # [max_imdb,   768]

            # Padding masks — True where padded
            c_pad = torch.zeros(max_critic_reviews, dtype=torch.bool)
            i_pad = torch.zeros(max_imdb_reviews,   dtype=torch.bool)
            c_pad[len(critic_texts):] = True
            i_pad[len(imdb_texts):]   = True

            cache[year].append({
                "film":       film,
                "critic_emb": c_emb.cpu(),   # [max_critic, 768]
                "imdb_emb":   i_emb.cpu(),   # [max_imdb,   768]
                "critic_pad": c_pad,         # [max_critic]
                "imdb_pad":   i_pad,         # [max_imdb]
                "winner_idx": winner_idx,
            })

            print(f"  {film}: {len(critic_texts)} critic, {len(imdb_texts)} IMDb reviews")

    torch.save(cache, save_path)
    print(f"\nSaved to {save_path}")
    print(f"Years cached: {sorted(cache.keys())}")
    return cache


def pad_embeddings(emb_tensor, target_len):
    """
    Pad embedding tensor to target_len with zero vectors.
    emb_tensor: [n, 768]
    returns:    [target_len, 768]
    """
    n, dim = emb_tensor.shape
    if n >= target_len:
        return emb_tensor[:target_len]
    pad = torch.zeros(target_len - n, dim, device=emb_tensor.device)
    return torch.cat([emb_tensor, pad], dim=0)


# ── SIMILARITY CHECK ──────────────────────────────────────────────────────────
def check_st_similarities(cache, year=2012):
    """
    Run this immediately after precompute to verify
    embeddings are now discriminative.
    Target: similarities in 0.60-0.85 range.
    """
    cohort = cache[year]
    vecs   = [c["critic_emb"][~c["critic_pad"]].mean(dim=0) for c in cohort]

    print(f"\nSentence transformer critic similarities ({year} cohort):")
    for i in range(len(cohort)):
        for j in range(i+1, len(cohort)):
            sim = torch.nn.functional.cosine_similarity(
                vecs[i].unsqueeze(0),
                vecs[j].unsqueeze(0)
            ).item()
            print(f"  {cohort[i]['film']:35s} vs {cohort[j]['film']:35s} : {sim:.4f}")


# ── CACHED DATASET ────────────────────────────────────────────────────────────
class OscarEmbeddingDataset(Dataset):
    """
    Loads precomputed embeddings instead of raw text.
    Each item is one cohort year — [n_films, max_reviews, 768] tensors.
    Training epochs are now seconds not minutes.
    """
    def __init__(self, cache, years):
        self.cohorts = []

        for year in years:
            if year not in cache:
                print(f"Warning: year {year} not in cache, skipping")
                continue

            film_data  = cache[year]
            n_films    = len(film_data)
            winner_idx = film_data[0]["winner_idx"]  # same for all films in year

            critic_embs = torch.stack([f["critic_emb"] for f in film_data])  # [n_films, max_critic, 768]
            imdb_embs   = torch.stack([f["imdb_emb"]   for f in film_data])  # [n_films, max_imdb,   768]
            critic_pads = torch.stack([f["critic_pad"]  for f in film_data])  # [n_films, max_critic]
            imdb_pads   = torch.stack([f["imdb_pad"]    for f in film_data])  # [n_films, max_imdb]
            films       = [f["film"] for f in film_data]

            self.cohorts.append({
                "year":        year,
                "films":       films,
                "critic_embs": critic_embs,
                "imdb_embs":   imdb_embs,
                "critic_pads": critic_pads,
                "imdb_pads":   imdb_pads,
                "winner_idx":  torch.tensor(winner_idx, dtype=torch.long),
            })

    def __len__(self):
        return len(self.cohorts)

    def __getitem__(self, idx):
        return self.cohorts[idx]

In [ ]:
# ── RUN PRECOMPUTE ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

cache_st = precompute_st_embeddings(
    nominees_df    = nominees_df_filtered,
    metacritic_df  = metacritic_df,
    imdb_df        = imdb_df,
    max_critic_reviews = 40,
    max_imdb_reviews   = 50,
    save_path      = "/content/drive/MyDrive/oscar_st_embeddings.pt",
    device         = device
)

# Check similarities immediately
check_st_similarities(cache_st, year=2012)

In [ ]:
# ── EVALUATION ────────────────────────────────────────────────────────────────
def evaluate_results(results, model_name="Model", k=3):
    """
    Unified evaluation for all three models.
    Computes Top-1, Top-3, and MRR with random baselines.
    Requires results dicts to have 'probs' and 'films' keys.
    """
    n       = len(results)
    n_films = np.mean([len(r["films"]) for r in results])

    top1_correct = 0
    topk_correct = 0
    rrs          = []

    print(f"\n{'='*56}")
    print(f"{model_name} — Evaluation Summary")
    print(f"{'='*56}")

    for r in results:
        winner = r["winner"]
        films  = r["films"]
        probs  = r["probs"]

        ranked      = sorted(zip(films, probs), key=lambda x: x[1], reverse=True)
        ranked_films = [f for f, p in ranked]

        # Top-1
        in_top1 = (ranked_films[0] == winner)
        top1_correct += int(in_top1)

        # Top-K
        in_topk = winner in ranked_films[:k]
        topk_correct += int(in_topk)

        # Reciprocal rank
        rank = ranked_films.index(winner) + 1
        rrs.append(1.0 / rank)

        mark1 = "✓" if in_top1 else "✗"
        markk = "✓" if in_topk else "✗"
        print(f"  {r['test_year']} | Top-1 {mark1} | Top-{k} {markk} | "
              f"RR: {1.0/rank:.3f} | "
              f"predicted: {r['predicted']:30s} | actual: {winner}")

    mrr          = np.mean(rrs)
    random_top1  = 1 / n_films
    random_topk  = k / n_films
    random_mrr   = np.mean([1/i for i in range(1, int(n_films)+1)])

    print(f"\n  Top-1 accuracy : {top1_correct}/{n} = {top1_correct/n:.1%}  "
          f"(random: {random_top1:.1%})")
    print(f"  Top-{k} accuracy : {topk_correct}/{n} = {topk_correct/n:.1%}  "
          f"(random: {random_topk:.1%})")
    print(f"  MRR            : {mrr:.4f}  "
          f"(random: {random_mrr:.4f})")

    return {
        "top1": top1_correct / n,
        "top3": topk_correct / n,
        "mrr":  mrr,
    }

### Model 1
**Transformer with Cross-Attention between IMDb user + Metacritic critic reviews encoded on shared RoBERTa Base**

**Hypothesis:** domain-separated encoding with bidirectional
cross-attention between critic and audience discourse captures
complementary signals that improve Best Picture prediction

**Features**
- RoBERTa base: improved version of BERT with 10x training data and improved hyperparameter tuning & dynamic masking
- Segmented feature space of IMDb user reviews and Metacritic reviews
- Both review streams pass through 2 self-attention layers, allowing each review to update its representation by attending to all other reviews of the same film
- Multi-head
- Cross-attention: explore relation between IMDb user reviews and Metacritic critic reviews
- Fusion layers: difference and product instead of simple concatenation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader

# ── MODEL ─────────────────────────────────────────────────────────────────────
class OscarPredictorAttention(nn.Module):
    """
    Dual-stream model with multi-head self-attention and cross-attention.
    Takes precomputed sentence transformer embeddings directly.
    Reduced capacity + heavy regularization for small dataset (8 training examples).

    Pipeline per film:
      1. Sentence transformer embeddings [num_reviews, 768] passed in directly
      2. Self-attention across reviews within each stream
      3. Cross-attention between streams (critic ↔ IMDb)
      4. Mean pool reviews → one vector per stream
      5. Fusion → single film logit
    """
    def __init__(
        self,
        hidden_dim=768,
        num_heads=4,               # reduced from 8
        num_self_attn_layers=1,    # reduced from 2
        dropout=0.5                # increased from 0.3
    ):
        super(OscarPredictorAttention, self).__init__()

        self.hidden_dim = hidden_dim

        # ── Self-attention over reviews within each stream ────────────────────
        critic_self_attn_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 2,   # reduced from hidden_dim * 4
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )
        self.critic_self_attn = nn.TransformerEncoder(
            critic_self_attn_layer,
            num_layers=num_self_attn_layers
        )

        imdb_self_attn_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 2,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )
        self.imdb_self_attn = nn.TransformerEncoder(
            imdb_self_attn_layer,
            num_layers=num_self_attn_layers
        )

        # ── Cross-attention between streams ───────────────────────────────────
        # Q=critic, K=IMDb, V=IMDb
        self.cross_attn_critic_to_imdb = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        # Q=IMDb, K=critic, V=critic
        self.cross_attn_imdb_to_critic = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm_critic_self  = nn.LayerNorm(hidden_dim)
        self.norm_imdb_self    = nn.LayerNorm(hidden_dim)
        self.norm_critic_cross = nn.LayerNorm(hidden_dim)
        self.norm_imdb_cross   = nn.LayerNorm(hidden_dim)

        # ── Fusion ────────────────────────────────────────────────────────────
        # diff and product give explicit relationship signals between streams
        fusion_input_dim = hidden_dim * 4  # critic + imdb + diff + product

        self.scorer = nn.Sequential(
            nn.Linear(fusion_input_dim, 128),  # removed middle layer, smaller
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
            # no sigmoid — softmax applied across cohort at training time
        )

    def mean_pool_reviews(self, x):
        """Collapse [batch, num_reviews, 768] → [batch, 768]"""
        return x.mean(dim=1)

    def forward(
        self,
        critic_embs,    critic_pad_mask,
        imdb_embs,      imdb_pad_mask,
    ):
        """
        critic_embs    : [n_films, max_critic_reviews, 768]
        critic_pad_mask: [n_films, max_critic_reviews]  True = padded
        imdb_embs      : [n_films, max_imdb_reviews,   768]
        imdb_pad_mask  : [n_films, max_imdb_reviews]    True = padded
        returns        : [n_films, 1] raw logit
        """
        h_critic = critic_embs
        h_imdb   = imdb_embs

        # 1. Self-attention within each stream
        h_critic_self = self.critic_self_attn(
            h_critic, src_key_padding_mask=critic_pad_mask
        )
        h_critic = self.norm_critic_self(h_critic + h_critic_self)

        h_imdb_self = self.imdb_self_attn(
            h_imdb, src_key_padding_mask=imdb_pad_mask
        )
        h_imdb = self.norm_imdb_self(h_imdb + h_imdb_self)

        # 2. Cross-attention between streams
        h_critic_cross, _ = self.cross_attn_critic_to_imdb(
            query=h_critic, key=h_imdb, value=h_imdb,
            key_padding_mask=imdb_pad_mask
        )
        h_critic = self.norm_critic_cross(h_critic + h_critic_cross)

        h_imdb_cross, _ = self.cross_attn_imdb_to_critic(
            query=h_imdb, key=h_critic, value=h_critic,
            key_padding_mask=critic_pad_mask
        )
        h_imdb = self.norm_imdb_cross(h_imdb + h_imdb_cross)

        # 3. Pool to one vector per stream
        c_critic = self.mean_pool_reviews(h_critic)   # [n_films, 768]
        c_imdb   = self.mean_pool_reviews(h_imdb)     # [n_films, 768]

        # 4. Fusion
        diff    = c_critic - c_imdb
        product = c_critic * c_imdb
        fused   = torch.cat([c_critic, c_imdb, diff, product], dim=-1)  # [n_films, 3072]

        return self.scorer(fused)                      # [n_films, 1]


# ── TRAIN / EVAL ──────────────────────────────────────────────────────────────
def train_step_cached(model, optimizer, batch, device):
    model.train()
    optimizer.zero_grad()

    critic_embs = batch["critic_embs"].squeeze(0).to(device)
    imdb_embs   = batch["imdb_embs"].squeeze(0).to(device)
    critic_pads = batch["critic_pads"].squeeze(0).to(device)
    imdb_pads   = batch["imdb_pads"].squeeze(0).to(device)
    winner_idx  = batch["winner_idx"].squeeze().to(device)

    logits = model(critic_embs, critic_pads, imdb_embs, imdb_pads)

    loss = nn.CrossEntropyLoss()(
        logits.squeeze(-1).unsqueeze(0),
        winner_idx.unsqueeze(0)
    )

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    probs     = torch.softmax(logits.squeeze(-1), dim=0)
    predicted = probs.argmax().item()
    correct   = (predicted == winner_idx.item())

    return loss.item(), correct, probs.detach().cpu().tolist()


def eval_step_cached(model, batch, device):
    model.eval()
    with torch.no_grad():
        critic_embs = batch["critic_embs"].squeeze(0).to(device)
        imdb_embs   = batch["imdb_embs"].squeeze(0).to(device)
        critic_pads = batch["critic_pads"].squeeze(0).to(device)
        imdb_pads   = batch["imdb_pads"].squeeze(0).to(device)
        winner_idx  = batch["winner_idx"].squeeze().to(device)

        logits = model(critic_embs, critic_pads, imdb_embs, imdb_pads)

        loss = nn.CrossEntropyLoss()(
            logits.squeeze(-1).unsqueeze(0),
            winner_idx.unsqueeze(0)
        )

        probs     = torch.softmax(logits.squeeze(-1), dim=0)
        predicted = probs.argmax().item()
        correct   = (predicted == winner_idx.item())

    return loss.item(), correct, probs.detach().cpu().tolist()


# ── LOOCV WITH EARLY STOPPING ─────────────────────────────────────────────────
def run_loocv_cached(cache, nominees_df,
                     num_epochs=100, lr=1e-3,
                     patience=8, device="cpu"):

    all_years = sorted(cache.keys())
    results   = []

    for test_year in all_years:
        print(f"\n{'='*56}")
        print(f"Fold: test year = {test_year}")
        print(f"{'='*56}")

        train_years = [y for y in all_years if y != test_year]

        train_dataset = OscarEmbeddingDataset(cache, train_years)
        test_dataset  = OscarEmbeddingDataset(cache, [test_year])

        train_loader  = DataLoader(train_dataset, batch_size=1, shuffle=True)
        test_loader   = DataLoader(test_dataset,  batch_size=1, shuffle=False)

        model     = OscarPredictorAttention().to(device)
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=lr,
            weight_decay=0.1       # aggressive regularization
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=num_epochs
        )

        # Early stopping state
        best_loss   = float("inf")
        no_improve  = 0
        best_epoch  = 0

        for epoch in range(num_epochs):
            losses, corrects = [], []

            for batch in train_loader:
                loss, correct, _ = train_step_cached(
                    model, optimizer, batch, device
                )
                losses.append(loss)
                corrects.append(correct)

            scheduler.step()
            mean_loss = np.mean(losses)

            # Print every 10 epochs
            if (epoch + 1) % 10 == 0:
                print(f"  Epoch {epoch+1:03d} | loss {mean_loss:.4f} | "
                      f"train acc {sum(corrects)}/{len(corrects)}")

            # Early stopping
            if mean_loss < best_loss - 1e-4:
                best_loss  = mean_loss
                no_improve = 0
                best_epoch = epoch + 1
                # Save best weights
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f"  Early stopping at epoch {epoch+1} "
                          f"(best epoch {best_epoch}, loss {best_loss:.4f})")
                    break

        # Restore best weights before evaluation
        model.load_state_dict(best_state)

        # Evaluate on held-out year
        print(f"\n  Results for {test_year}:")
        for batch in test_loader:
            loss, correct, probs = eval_step_cached(model, batch, device)
            films      = batch["films"]
            winner_idx = batch["winner_idx"].squeeze().item()
            pred_idx   = probs.index(max(probs))

            for i, (film, prob) in enumerate(zip(films, probs)):
                w = " ← winner"    if i == winner_idx else ""
                p = " ← predicted" if i == pred_idx   else ""
                print(f"    {film[0]:45s} {prob:.3f}{w}{p}")

            results.append({
                "test_year": test_year,
                "correct":   correct,
                "winner":    films[winner_idx][0],
                "predicted": films[pred_idx][0],
                "probs":     probs,
                "films":     [f[0] for f in films],
            })

    # Summary
    print(f"\n{'='*56}")
    print(f"LOOCV Summary")
    print(f"{'='*56}")
    for r in results:
        mark = "✓" if r["correct"] else "✗"
        print(f"  {mark} {r['test_year']} | "
              f"predicted: {r['predicted']:40s} | "
              f"actual: {r['winner']}")

    accuracy = sum(r["correct"] for r in results) / len(results)
    print(f"\nOverall: {sum(r['correct'] for r in results)}/{len(results)} = {accuracy:.1%}")
    print(f"Random baseline: ~12.5%")

    return results

In [ ]:
# ── RUN ───────────────────────────────────────────────────────────────────────
results_st = run_loocv_cached(
    cache       = cache_st,
    nominees_df = nominees_df_filtered,
    num_epochs  = 100,
    lr          = 1e-3,
    patience    = 8,
    device      = device
)
eval_attn = evaluate_results(results_st, model_name="Model 1 — Attention + Cross-Attention", k=3)

### Model 2
**Baseline BERT language model**

**Hypothesis:** BERT's pretrained English understanding alone,
without architectural sophistication, provides meaningful signal
above a non-contextual baseline

**Features:**

- One shared encoder: no domain separation between critic and IMDb
- Mean pooling as entire aggregation strategy: every review weighted equally, no attention
- Simple concatenation: no diff, no product, no cross-stream interaction
- One MLP head instead of 8 that learn different review aspects that relate to a Best Picture win (e.g. emotions in writing, logic, commentary related to technical aspects of film)

In [ ]:
# Model 2 hyperparameter tuning
# Expanded search over Model 2 training and MLP-capacity hyperparameters.
# Each sampled configuration is evaluated across multiple seeds for stability.

import itertools
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader


def set_random_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class OscarPredictorSimple(nn.Module):
    """Self-contained Model 2 definition for hyperparameter tuning."""
    def __init__(self, hidden_dim=768, mlp_hidden_dim=64, dropout=0.5):
        super().__init__()
        fusion_input_dim = hidden_dim * 4
        self.scorer = nn.Sequential(
            nn.Linear(fusion_input_dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden_dim, 1),
        )

    def masked_mean(self, embs, pad_mask):
        real_mask = (~pad_mask).float().unsqueeze(-1)
        return (embs * real_mask).sum(dim=1) / real_mask.sum(dim=1).clamp(min=1e-8)

    def forward(self, critic_embs, critic_pad_mask, imdb_embs, imdb_pad_mask):
        c_critic = self.masked_mean(critic_embs, critic_pad_mask)
        c_imdb = self.masked_mean(imdb_embs, imdb_pad_mask)
        diff = c_critic - c_imdb
        product = c_critic * c_imdb
        fused = torch.cat([c_critic, c_imdb, diff, product], dim=-1)
        return self.scorer(fused)


def train_step_simple(model, optimizer, batch, device, label_smoothing=0.0):
    model.train()
    optimizer.zero_grad()

    critic_embs = batch["critic_embs"].squeeze(0).to(device)
    imdb_embs = batch["imdb_embs"].squeeze(0).to(device)
    critic_pads = batch["critic_pads"].squeeze(0).to(device)
    imdb_pads = batch["imdb_pads"].squeeze(0).to(device)
    winner_idx = batch["winner_idx"].squeeze().to(device)

    logits = model(critic_embs, critic_pads, imdb_embs, imdb_pads)
    loss = nn.CrossEntropyLoss(label_smoothing=label_smoothing)(
        logits.squeeze(-1).unsqueeze(0),
        winner_idx.unsqueeze(0),
    )

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    probs = torch.softmax(logits.squeeze(-1), dim=0)
    predicted = probs.argmax().item()
    correct = predicted == winner_idx.item()
    return loss.item(), correct, probs.detach().cpu().tolist()


def eval_step_simple(model, batch, device, label_smoothing=0.0):
    model.eval()
    with torch.no_grad():
        critic_embs = batch["critic_embs"].squeeze(0).to(device)
        imdb_embs = batch["imdb_embs"].squeeze(0).to(device)
        critic_pads = batch["critic_pads"].squeeze(0).to(device)
        imdb_pads = batch["imdb_pads"].squeeze(0).to(device)
        winner_idx = batch["winner_idx"].squeeze().to(device)

        logits = model(critic_embs, critic_pads, imdb_embs, imdb_pads)
        loss = nn.CrossEntropyLoss(label_smoothing=label_smoothing)(
            logits.squeeze(-1).unsqueeze(0),
            winner_idx.unsqueeze(0),
        )

        probs = torch.softmax(logits.squeeze(-1), dim=0)
        predicted = probs.argmax().item()
        correct = predicted == winner_idx.item()

    return loss.item(), correct, probs.detach().cpu().tolist()


def make_model2_param_grid(grid):
    keys = list(grid.keys())
    return [
        dict(zip(keys, values))
        for values in itertools.product(*(grid[key] for key in keys))
    ]


def sample_model2_param_grid(grid, n_configs=24, seed=11):
    """Sample a manageable number of configs from a larger Cartesian grid."""
    all_configs = make_model2_param_grid(grid)
    rng = random.Random(seed)
    rng.shuffle(all_configs)
    return all_configs[:min(n_configs, len(all_configs))]


def run_loocv_simple_tuned(
    cache,
    nominees_df,
    num_epochs=100,
    lr=1e-3,
    patience=8,
    dropout=0.5,
    weight_decay=0.1,
    mlp_hidden_dim=64,
    label_smoothing=0.0,
    seed=42,
    device="cpu",
    verbose=False,
):
    """
    Same LOOCV logic as run_loocv_simple, but exposes regularization,
    optimization, seed, and small MLP-capacity choices.
    """
    all_years = sorted(cache.keys())
    results = []

    for test_year in all_years:
        set_random_seed(seed + int(test_year))
        if verbose:
            print(f"\n{'='*56}")
            print(f"Fold: test year = {test_year}  [Simple MLP]")
            print(f"{'='*56}")

        train_years = [y for y in all_years if y != test_year]

        train_dataset = OscarEmbeddingDataset(cache, train_years)
        test_dataset = OscarEmbeddingDataset(cache, [test_year])

        generator = torch.Generator().manual_seed(seed + int(test_year))
        train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, generator=generator)
        test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

        model = OscarPredictorSimple(
            dropout=dropout,
            mlp_hidden_dim=mlp_hidden_dim,
        ).to(device)
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay,
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=num_epochs
        )

        best_loss = float("inf")
        no_improve = 0
        best_epoch = 0
        best_state = None

        for epoch in range(num_epochs):
            losses, corrects = [], []

            for batch in train_loader:
                loss, correct, _ = train_step_simple(
                    model, optimizer, batch, device, label_smoothing=label_smoothing
                )
                losses.append(loss)
                corrects.append(correct)

            scheduler.step()
            mean_loss = np.mean(losses)

            if verbose and (epoch + 1) % 10 == 0:
                print(f"  Epoch {epoch+1:03d} | loss {mean_loss:.4f} | "
                      f"train acc {sum(corrects)}/{len(corrects)}")

            if mean_loss < best_loss - 1e-4:
                best_loss = mean_loss
                no_improve = 0
                best_epoch = epoch + 1
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                no_improve += 1
                if no_improve >= patience:
                    if verbose:
                        print(f"  Early stopping at epoch {epoch+1} "
                              f"(best epoch {best_epoch}, loss {best_loss:.4f})")
                    break

        if best_state is not None:
            model.load_state_dict(best_state)

        for batch in test_loader:
            loss, correct, probs = eval_step_simple(
                model, batch, device, label_smoothing=label_smoothing
            )
            films = batch["films"]
            winner_idx = batch["winner_idx"].squeeze().item()
            pred_idx = probs.index(max(probs))

            if verbose:
                print(f"\n  Results for {test_year}:")
                for i, (film, prob) in enumerate(zip(films, probs)):
                    w = " <- winner" if i == winner_idx else ""
                    p = " <- predicted" if i == pred_idx else ""
                    print(f"    {film[0]:45s} {prob:.3f}{w}{p}")

            results.append({
                "test_year": test_year,
                "correct": correct,
                "loss": loss,
                "winner": films[winner_idx][0],
                "predicted": films[pred_idx][0],
                "probs": probs,
                "films": [f[0] for f in films],
                "best_epoch": best_epoch,
            })

    return results


def count_top_k(results, k=3):
    top_k_correct = 0

    for r in results:
        ranked = sorted(zip(r["films"], r["probs"]), key=lambda x: x[1], reverse=True)
        top_k_films = [film for film, prob in ranked[:k]]
        top_k_correct += int(r["winner"] in top_k_films)

    return top_k_correct


def tune_model2_hyperparameters(cache, nominees_df, param_grid, device, k=3, seeds=(13, 37, 101)):
    tuning_rows = []

    for config_idx, params in enumerate(param_grid, start=1):
        print(f"\nConfig {config_idx}/{len(param_grid)}: {params}")

        seed_summaries = []
        seed_results = {}

        for seed in seeds:
            results = run_loocv_simple_tuned(
                cache=cache,
                nominees_df=nominees_df,
                device=device,
                verbose=False,
                seed=seed,
                **params,
            )

            top1_correct = sum(r["correct"] for r in results)
            topk_correct = count_top_k(results, k=k)
            mean_loss = np.mean([r["loss"] for r in results])
            mean_best_epoch = np.mean([r["best_epoch"] for r in results])

            seed_summaries.append({
                "seed": seed,
                "top1_correct": top1_correct,
                f"top{k}_correct": topk_correct,
                "mean_loss": mean_loss,
                "mean_best_epoch": mean_best_epoch,
            })
            seed_results[seed] = results

        top1_values = [row["top1_correct"] for row in seed_summaries]
        topk_values = [row[f"top{k}_correct"] for row in seed_summaries]
        loss_values = [row["mean_loss"] for row in seed_summaries]
        epoch_values = [row["mean_best_epoch"] for row in seed_summaries]

        best_seed_row = max(
            seed_summaries,
            key=lambda row: (row["top1_correct"], row[f"top{k}_correct"], -row["mean_loss"]),
        )
        best_seed = best_seed_row["seed"]

        print(f"  Avg Top-1: {np.mean(top1_values):.2f}/9 ± {np.std(top1_values):.2f} | "
              f"Avg Top-{k}: {np.mean(topk_values):.2f}/9 ± {np.std(topk_values):.2f} | "
              f"Mean loss: {np.mean(loss_values):.4f}")

        tuning_rows.append({
            **params,
            "top1_correct": np.mean(top1_values),
            "top1_std": np.std(top1_values),
            f"top{k}_correct": np.mean(topk_values),
            f"top{k}_std": np.std(topk_values),
            "mean_loss": np.mean(loss_values),
            "mean_best_epoch": np.mean(epoch_values),
            "best_seed": best_seed,
            "seed_summaries": seed_summaries,
            "results": seed_results[best_seed],
        })

    tuning_df = pd.DataFrame(tuning_rows).sort_values(
        by=["top1_correct", f"top{k}_correct", "mean_loss"],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    return tuning_df


# Larger search space, sampled to keep runtime manageable.
# 24 sampled configs x 3 seeds = 72 LOOCV runs.
model2_search_space = {
    "num_epochs": [80, 120, 160],
    "lr": [1e-4, 3e-4, 7e-4, 1e-3],
    "patience": [8, 12, 16],
    "dropout": [0.2, 0.3, 0.5, 0.65],
    "weight_decay": [0.0, 0.01, 0.05, 0.1],
    "mlp_hidden_dim": [32, 64, 128, 256],
    "label_smoothing": [0.0, 0.03, 0.07],
}

model2_param_grid = sample_model2_param_grid(
    model2_search_space,
    n_configs=24,
    seed=11,
)

tuning_df_simple = tune_model2_hyperparameters(
    cache=cache_st,
    nominees_df=nominees_df_filtered,
    param_grid=model2_param_grid,
    device=device,
    k=3,
    seeds=(13, 37, 101),
)

display_cols = [
    "top1_correct",
    "top1_std",
    "top3_correct",
    "top3_std",
    "mean_loss",
    "mean_best_epoch",
    "best_seed",
    "num_epochs",
    "lr",
    "patience",
    "dropout",
    "weight_decay",
    "mlp_hidden_dim",
    "label_smoothing",
]
display(tuning_df_simple[display_cols])

best_model2_params = {
    key: tuning_df_simple.loc[0, key]
    for key in ["num_epochs", "lr", "patience", "dropout", "weight_decay", "mlp_hidden_dim", "label_smoothing", "best_seed"]
}
print("\nBest Model 2 hyperparameters:")
print(best_model2_params)

results_simple_tuned = tuning_df_simple.loc[0, "results"]
evaluate_results(results_simple_tuned, model_name="Model 2 — Tuned MLP", k=3)

### Model 3: TF-IDF Baseline

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
import scipy.sparse as sp


# ── TEXT PREPARATION ──────────────────────────────────────────────────────────
def prepare_texts(nominees_df, metacritic_df, imdb_df, year):
    """
    For a given year, concatenate all critic reviews and all IMDb reviews
    into one string per film. Returns list of (film, critic_text, imdb_text, is_winner).
    """
    year_nominees = nominees_df[nominees_df["ceremony_year"] == year]
    films         = year_nominees["film_title"].tolist()
    winner_row    = year_nominees[year_nominees["winner"] == 1]
    winner_title  = winner_row["film_title"].iloc[0] if not winner_row.empty else None

    film_data = []

    for film in films:
        # Concatenate all critic quotes into one string
        critic_texts = metacritic_df[
            (metacritic_df["film_title"]    == film) &
            (metacritic_df["ceremony_year"] == year)
        ]["clean_text"].dropna().tolist()

        # Concatenate all IMDb reviews into one string
        imdb_texts = imdb_df[
            (imdb_df["film_title"]    == film) &
            (imdb_df["ceremony_year"] == year)
        ]["clean_text"].dropna().tolist()

        critic_combined = " ".join(critic_texts) if critic_texts else "no reviews"
        imdb_combined   = " ".join(imdb_texts)   if imdb_texts   else "no reviews"
        is_winner       = 1 if film == winner_title else 0

        film_data.append({
            "film":          film,
            "critic_text":   critic_combined,
            "imdb_text":     imdb_combined,
            "is_winner":     is_winner,
        })

    return film_data


# ── TFIDF MODEL ───────────────────────────────────────────────────────────────
class OscarPredictorTFIDF:
    """
    Model 3 — TF-IDF + Logistic Regression.
    Non-neural baseline. No embeddings, no training loop, no GPU.

    Critic and IMDb reviews are each vectorized with separate TF-IDF
    vectorizers then concatenated into one feature vector per film.
    Logistic regression predicts P(winner).

    Purpose: sanity check baseline.
    If neural models don't beat this, something is wrong with the
    neural pipeline or the data is too noisy for any model.
    """
    def __init__(self, max_features=5000, C=0.01):
        # Separate vectorizers — critic and audience use different vocabulary
        # min_df=1 because corpus is tiny (77 films total)
        self.critic_vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(1, 2),      # unigrams + bigrams
            sublinear_tf=True,       # log normalization on term frequency
            strip_accents="unicode",
            min_df=1,
        )
        self.imdb_vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(1, 2),
            sublinear_tf=True,
            strip_accents="unicode",
            min_df=1,
        )
        # C=0.01 — heavy regularization essential for small dataset
        # class_weight="balanced" — corrects 1 winner vs 8 nominees imbalance
        self.classifier = LogisticRegression(
            C=C,
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )

    def _build_features(self, film_data_list, fit=False):
        """
        Vectorize critic and IMDb texts and concatenate.
        fit=True during training, fit=False during test (transform only).

        returns: sparse matrix [n_films, max_features * 2]
        """
        critic_texts = [d["critic_text"] for d in film_data_list]
        imdb_texts   = [d["imdb_text"]   for d in film_data_list]

        if fit:
            critic_features = self.critic_vectorizer.fit_transform(critic_texts)
            imdb_features   = self.imdb_vectorizer.fit_transform(imdb_texts)
        else:
            critic_features = self.critic_vectorizer.transform(critic_texts)
            imdb_features   = self.imdb_vectorizer.transform(imdb_texts)

        # Horizontally stack critic and IMDb features
        return sp.hstack([critic_features, imdb_features])

    def fit(self, film_data_list):
        """Train on a list of film dicts (from multiple years)."""
        X = self._build_features(film_data_list, fit=True)
        y = np.array([d["is_winner"] for d in film_data_list])
        self.classifier.fit(X, y)

    def predict_proba(self, film_data_list):
        """
        Returns P(winner) for each film.
        Normalized to sum to 1 across cohort for comparability
        with softmax outputs from neural models.
        """
        X     = self._build_features(film_data_list, fit=False)
        probs = self.classifier.predict_proba(X)[:, 1]  # P(winner)

        # Normalize across cohort — sum to 1
        total = probs.sum()
        if total > 0:
            probs = probs / total

        return probs


# ── LOOCV ─────────────────────────────────────────────────────────────────────
def run_loocv_tfidf(nominees_df, metacritic_df, imdb_df,
                    max_features=5000, C=0.01):
    """
    Leave-one-year-out cross validation for TF-IDF model.
    Fits vectorizer and classifier on training years only.
    Transforms test year using training vocabulary — no leakage.
    """
    all_years = sorted(nominees_df["ceremony_year"].unique())
    results   = []

    for test_year in all_years:
        print(f"\n{'='*56}")
        print(f"Fold: test year = {test_year}  [TF-IDF]")
        print(f"{'='*56}")

        train_years = [y for y in all_years if y != test_year]

        # Build training data — all films from all training years
        train_data = []
        for year in train_years:
            train_data.extend(prepare_texts(nominees_df, metacritic_df, imdb_df, year))

        # Build test data — films from held-out year only
        test_data = prepare_texts(nominees_df, metacritic_df, imdb_df, test_year)

        print(f"  Train: {len(train_data)} films ({sum(d['is_winner'] for d in train_data)} winners)")
        print(f"  Test:  {len(test_data)} films")

        # Fit on training data, transform test data
        model = OscarPredictorTFIDF(max_features=max_features, C=C)
        model.fit(train_data)
        probs = model.predict_proba(test_data)

        # Find winner and prediction
        true_idx = next(i for i, d in enumerate(test_data) if d["is_winner"] == 1)
        pred_idx = probs.argmax()
        correct  = (pred_idx == true_idx)
        films    = [d["film"] for d in test_data]

        # Print results
        print(f"\n  Results for {test_year}:")
        for i, (film, prob) in enumerate(zip(films, probs)):
            w = " ← winner"    if i == true_idx else ""
            p = " ← predicted" if i == pred_idx else ""
            print(f"    {film:45s} {prob:.3f}{w}{p}")

        results.append({
            "test_year": test_year,
            "correct":   correct,
            "winner":    films[true_idx],
            "predicted": films[pred_idx],
            "probs":     probs.tolist(),
            "films":     films,
        })

    # Summary
    print(f"\n{'='*56}")
    print(f"TF-IDF LOOCV Summary")
    print(f"{'='*56}")
    for r in results:
        mark = "✓" if r["correct"] else "✗"
        print(f"  {mark} {r['test_year']} | "
              f"predicted: {r['predicted']:40s} | "
              f"actual: {r['winner']}")

    accuracy = sum(r["correct"] for r in results) / len(results)
    print(f"\nOverall: {sum(r['correct'] for r in results)}/{len(results)} = {accuracy:.1%}")
    print(f"Random baseline: ~12.5%")

    return results

In [ ]:
results_tfidf = run_loocv_tfidf(
    nominees_df   = nominees_df_filtered,
    metacritic_df = metacritic_df,
    imdb_df       = imdb_df,
    max_features  = 5000,
    C             = 20.0
)
evaluate_results(results_tfidf, model_name="Model 3 — TF-IDF", k=3)